In [41]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [42]:
import torch
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
import tqdm 
from omegaconf import OmegaConf

import what_where as ww

In [43]:
cfg = ww.utils.load_config("config_contrast_detection")

results_dir = ww.utils.RESULTS_DIR / cfg.experiment.name
results_dir.mkdir(parents=True, exist_ok=True)

cfg.train.instance = 0

In [44]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# loading the latest checkpoint
checkpoint_path = ww.utils.get_checkpoint_path(cfg)
print("checkpoint file: ", checkpoint_path)
# checkpoint_path = get_best_checkpoint_from_cfg(cfg)
if checkpoint_path is None:
    raise ValueError("No checkpoint found for config")
checkpoint = torch.load(checkpoint_path, weights_only=False)

# loading the config from the checkpoint
checkpoint_cfg = OmegaConf.create(checkpoint["config"])
# checkpoint_cfg.train.energy.st_sample_ratio = 1.0

# readout = ww.model.GratingsReadout(checkpoint_cfg)
model = ww.model.Model(checkpoint_cfg)

dataset = ww.utils.get_datasets(cfg)["valid"]
dataloader = DataLoader(dataset, batch_size=cfg.train.dataloader.batch_size, shuffle=True)

# loading model weights
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()

/home/eivinas/dev/what-where/checkpoints/contrast_detection/contrast_detection/ean_space/instance0
checkpoint:  /home/eivinas/dev/what-where/checkpoints/contrast_detection/contrast_detection/ean_space/instance0/checkpoint_030.pth 

checkpoint file:  /home/eivinas/dev/what-where/checkpoints/contrast_detection/contrast_detection/ean_space/instance0/checkpoint_030.pth


Model(
  (cnn): CNN(
    (conv_layers): ModuleList(
      (0): CNNLayer(
        (conv_layer): Conv2d(1, 64, kernel_size=(5, 5), stride=(2, 2), padding=(2, 2))
        (divisive_normalization): DivisiveNormalizationConv()
      )
      (1): CNNLayer(
        (conv_layer): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (divisive_normalization): DivisiveNormalizationConv()
      )
      (2): CNNLayer(
        (conv_layer): Conv2d(128, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (divisive_normalization): DivisiveNormalizationConv()
      )
    )
  )
  (rnn): RNN(
    (rnn_layers): ModuleList(
      (0): RNNLayer(
        (input_to_hidden): Linear(in_features=16384, out_features=256, bias=True)
        (hidden_to_hidden): Linear(in_features=256, out_features=256, bias=True)
        (divisive_normalization): DivisiveNormalizationRNN()
      )
      (1-2): 2 x RNNLayer(
        (input_to_hidden): Linear(in_features=256, out_features=256, bias=T

In [45]:

n_dataset = len(dataset)
n_passes = checkpoint_cfg.model.n_passes
# conv_channels = checkpoint_cfg.model.cnn.conv_channels
last_pass = n_passes - 1    

n_rnn_layers = cfg.model.rnn.n_layers
hidden_size = cfg.model.rnn.hidden_size

activations = {}
for conv_layer in ["conv1", "conv3"]:
    activations[conv_layer] = {}
    for side in ["left", "right"]:
        n_channels = 64 if conv_layer == "conv1" else 256
        activations[conv_layer][side] = torch.zeros((n_dataset, cfg.experiment.n_reps, len(cfg.experiment.gain_suppression_amounts), len(cfg.experiment.energy_costs), n_passes, n_channels))


def get_gratings_data(model, dataloader):
    data = {
        "orientation_left" : [],
        "orientation_right" : [],
        "contrast_left" : [],
        "contrast_right" : [],
        "present_left" : [],
        "present_right" : [],
        "attend_left" : [],
        "present_pred" : [],
        "accuracy" : [],
        "gain_suppression" : [], # this is the gain suppression amount chosen
        "gain_suppression_sampled" : [], # this is the actual sampled gain suppression (can be either 0.0 or the gain suppression amount chosen)
        "stimulus_onset" : [],
        "energy_cost" : [],
        "energy_use" : [],
        "t" : [],
        "repetition" : [],
        "trial" : [],
    }

    for rep in range(cfg.experiment.n_reps):
        print(f"Processing repetition {rep + 1}/{cfg.experiment.n_reps}")

        for i, (images, present, meta_info) in enumerate(tqdm.tqdm(dataloader, desc="Processing batches")):
            images = images.to(device)

            start_index = i * cfg.train.dataloader.batch_size
            end_index = start_index + images.shape[0]

            for j, gain_suppression_amount in enumerate(cfg.experiment.gain_suppression_amounts):
                for k, energy_cost in enumerate(cfg.experiment.energy_costs):

                    log_energy_cost = torch.tensor(energy_cost).to(device).repeat(images.shape[0], 1)
                    gain_suppression = ww.utils.sample_gain_suppression(cfg, images.shape[0], n_passes,
                                                                        meta_info["stimulus_onset"],
                                                                        device, amount=gain_suppression_amount)

                    with torch.no_grad():
                        out_full = model(images, log_energy_cost, n_passes, gain_suppression=gain_suppression, noise_anneal=1.0)

                    # accuracy
                    for t in range(n_passes):
                        for key in data.keys():
                            if key in meta_info:
                                data[key].extend(meta_info[key].cpu().numpy())

                        # gain suppression
                        batch_indices = torch.arange(gain_suppression["conv1"].shape[0])
                        data["gain_suppression_sampled"].extend(gain_suppression["conv1"][batch_indices, meta_info["stimulus_onset"]].cpu().numpy())
                        data["gain_suppression"].extend([gain_suppression_amount] * images.shape[0])
                        data["energy_cost"].extend([energy_cost] * images.shape[0])

                        predicted_present = torch.softmax(out_full[t]["prediction"]["what"], dim=1).cpu()
                        data["present_pred"].extend(predicted_present[:, 1].cpu().numpy())  # probability of being present
                        accuracies = (predicted_present.argmax(dim=1) == present["what"]).float()
                        data["accuracy"].extend(accuracies.cpu().numpy())

                        # pass energy
                        energy_use = torch.zeros((images.shape[0]), device=device)
                        energy_use_t = ww.utils.get_energy_use(cfg, out_full, t, device)
                        for key in energy_use_t.keys():
                            energy_use += energy_use_t[key]

                        data["energy_use"].extend(energy_use.cpu().numpy())
                        data["t"].extend([t] * images.shape[0])
                        data["trial"].extend(range(start_index, end_index))
                        data["repetition"].extend([rep] * images.shape[0])

                        # activations
                        for (side_index, side) in enumerate(["left", "right"]):
                            for (conv_layer_index, conv_layer) in enumerate(["conv1", "conv3"]):
                                pos = list(cfg.dataset.center_left) if side == "left" else list(cfg.dataset.center_right)
                                ratio = ratio = 64 / 32 if conv_layer == "conv1" else 64 / 8
                                unit_index = [int(pos[0] / ratio), int(pos[1] / ratio)]  # convert to conv1 receptive field coordinates
                                # activation = out_full[t]["activations"][conv_layer][:,:,*unit_index].cpu().numpy()
                                activation = out_full[t]["activations"][conv_layer][:,:,*unit_index]
                                activations[conv_layer][side][start_index:end_index, rep, j, k, t, :] = activation.cpu()
            # break
    print("creating the dataframe")
    df_full = pd.DataFrame(data)
    print("dataframe created with shape:", df_full.shape)
    return df_full, activations


results_path = results_dir / f"{cfg.experiment.name}_results_data.parquet"
activations_path = results_dir / f"{cfg.experiment.name}_activations.npy"

if results_path.exists() and activations_path.exists() and not cfg.experiment.redo_extraction:
    print("Results data and activations already exist, loading them...")
    df_full = pd.read_parquet(results_path)
    activations_np = np.load(activations_path, allow_pickle=True).item()
    for conv_layer in activations.keys():
        for side in activations[conv_layer].keys():
            activations[conv_layer][side] = torch.tensor(activations_np[conv_layer][side])
    print("Data and activations loaded.")

else:
    print("Extracting results data and activations...")
    df_full, activations = get_gratings_data(model, dataloader)
    print("Saving results data and activations...")
    df_full.to_parquet(results_path, index=False)
    activations_np = {}
    for conv_layer in activations.keys():
        activations_np[conv_layer] = {}
        for side in activations[conv_layer].keys():
            activations_np[conv_layer][side] = activations[conv_layer][side].cpu().numpy()
    np.save(activations_path, activations_np)
    print("Results data and activations saved.")


Extracting results data and activations...
Processing repetition 1/3


Processing batches: 100%|██████████| 79/79 [01:30<00:00,  1.14s/it]


Processing repetition 2/3


Processing batches: 100%|██████████| 79/79 [01:30<00:00,  1.14s/it]


Processing repetition 3/3


Processing batches: 100%|██████████| 79/79 [01:30<00:00,  1.15s/it]


creating the dataframe
dataframe created with shape: (2160000, 17)
Saving results data and activations...
Results data and activations saved.
